In [ ]:
schema_list = ['dbo'] # Pass Schema Lists whose Tables shortcut has to be created

In [ ]:
%%tsql -artifact Warehouse_Name -type Warehouse -session -bind df
SELECT TABLE_NAME,TABLE_SCHEMA FROM INFORMATION_SCHEMA.TABLES

In [ ]:
df = df[df["TABLE_SCHEMA"].isin(schema_list)]

In [ ]:
import requests
import json
from azure.identity import ClientSecretCredential

#SP Creds
TENANT_ID = ""
CLIENT_ID = ""
CLIENT_SECRET = ""
WORKSPACE_ID = "" # Workspace ID From URL
SCOPE = "https://api.fabric.microsoft.com/.default"

credential = ClientSecretCredential(tenant_id=TENANT_ID, client_id=CLIENT_ID, client_secret=CLIENT_SECRET)
token = credential.get_token(SCOPE).token
print("Token acquired")

In [ ]:
def create_warehouse_shortcut(WAREHOUSE_TABLE,LAKEHOUSE_TARGET,TABLE_NAME):
    url = f"https://api.fabric.microsoft.com/v1/workspaces/{WORKSPACE_ID}/items/{LAKEHOUSE_ID}/shortcuts"

    payload =  {
    "path": LAKEHOUSE_TARGET,
    "name": TABLE_NAME,
    "target": {
        "oneLake": {
        "workspaceId": WORKSPACE_ID,
        "itemId": WAREHOUSE_ID,
        "path": WAREHOUSE_TABLE
        }
    }
    }
    
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }
    
    print(f"\n=== Creating shortcut ===")
    print(f"From: {WAREHOUSE_ID}/Tables/{WAREHOUSE_TABLE}")
    print(f"To:   {LAKEHOUSE_ID}/{LAKEHOUSE_TARGET}")
    
    try:
        response = requests.post(url, headers=headers, json=payload)
        print(f"HTTP Status: {response.status_code}")
        
        if response.status_code in [200, 201, 202,409]:
            result = response.json()
            print("SHORTCUT CREATED SUCCESSFULLY!")
            # print("Response:", json.dumps(result, indent=2))
            return True
        else:
            print("ERROR:")
            print("Status:", response.status_code)
            print("Body:", response.text[:500])
            return False
            
    except Exception as e:
        print(f"Exception: {e}")
        return False


In [ ]:
WAREHOUSE_ID = "" # Warehouse ID in which Tables actually reside in
LAKEHOUSE_ID = "" # Lakehouse ID in which Tables Shortcuts have to be created in
WORKSPACE_ID = "" # Workspace ID From URL
if __name__ == "__main__":
    for schema, table in zip(df["TABLE_SCHEMA"], df["TABLE_NAME"]):
        if schema in schema_list:
            print(f"{schema}.{table}")
            WAREHOUSE_TABLE = f"Tables/{schema}/{table}"
            LAKEHOUSE_TARGET = f"Tables/{schema}"
            TABLE_NAME = table
            success = create_warehouse_shortcut(WAREHOUSE_TABLE,LAKEHOUSE_TARGET,TABLE_NAME)
        
            if success:
                print(f'Shortcut Created for Table - {schema}.{table}')